In [ ]:
import mario
import yaml
import pandas as pd
import os

import warnings
warnings.filterwarnings("ignore")

user = 'LR'   # change this to your username
years = range(2025,2026)   
versions = [
    # 'v1.0', 
    'v2.0'
    ]  # versions to parse

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

In [ ]:
footprints = pd.DataFrame()
ghgs = [
    "Carbon dioxide, fossil (air - Emiss)",
    "CH4 (air - Emiss)",
    "N2O (air - Emiss)",
]

for version in versions:
    for year in years:
        print(f"Parsing version {version} for year {year}...", end="\n")
        db = mario.parse_from_txt(
            path = os.path.join(paths['export'], version, str(year),"flows"),
            mode = "flows",
            table = 'SUT',
        )

        print("DONE! Calculating footprints...", end="\n")
        f = db.f.loc[ghgs,:]
        f = f.T
        f['GHG'] = f['Carbon dioxide, fossil (air - Emiss)'] + \
                   f['CH4 (air - Emiss)']*25 + \
                   f['N2O (air - Emiss)']*298

        f.columns.names = ['Substances']
        f = f.stack().to_frame()
        f.columns = ['Value']
        f.reset_index(inplace=True)
        f['Year'] = year
        f['Version'] = version

        footprints = pd.concat([footprints, f], axis=0, ignore_index=True)
        print("DONE!\n")

Add footprints from EXIOBASE 3.3.18 raw

In [ ]:
version = "EXIOBASE 3.3.18"
year = 2011


print(f"Parsing version {version} for year {year}...", end="\n")
db = mario.parse_from_txt(
    paths['raw'], 
    table='SUT', 
    mode='flows'
)


print("DONE! Aggregating EE...", end="\n")
db.aggregate("support/aggregate_ee.xlsx",ignore_nan=True)


print("DONE! Calculating footprints...", end="\n")
f = db.f.loc[ghgs,:]
f = f.T
f['GHG'] = f['Carbon dioxide, fossil (air - Emiss)'] + \
            f['CH4 (air - Emiss)']*29.8 + \
            f['N2O (air - Emiss)']*273


f.columns.names = ['Substances']
f = f.stack().to_frame()
f.columns = ['Value']
f.reset_index(inplace=True)
f['Year'] = year
f['Version'] = version


footprints = pd.concat([footprints, f], axis=0, ignore_index=True)
print("DONE!\n")

In [ ]:
footprints.to_csv(
    paths['export']+"/_results/Footprints.csv",
    index=False
)

In [ ]:
footprints = pd.read_csv(paths['export']+"/_results/Footprints.csv") 

In [ ]:
footprints.head()

In [ ]:
footprints.query("" \
"Level=='Commodity' and " \
"Substances=='GHG' and " \
"Region=='IT' and "
"(Item=='Electricity' or Item=='Electricity need')" \
"").sort_values(by=['Year','Version'])